# CMAPSS full pipeline on Google Colab

**Run cells top → bottom.** One notebook does:

1. Clone this repo  
2. Install dependencies (GPU PyTorch if you use a GPU runtime)  
3. Upload NASA CMAPSS to Colab disk → import into project  
4. Phase 2 build + Phase 3 train → **MLflow logs to Databricks**  
5. Verification report  
6. Zip & download models, artifacts, predictions (runs stay in Databricks)  

**Before you start:** `Runtime` → `Change runtime type` → **T4 GPU** (optional; LSTM only).  
**Edit cell 2:** set `REPO_URL` to your GitHub repo (HTTPS).

In [ ]:
# ── CONFIG (edit these) ──────────────────────────────────────────────
REPO_URL = "https://github.com/YOUR_USERNAME/predictive-maintenance.git"  # required
REPO_DIR = "/content/predictive-maintenance"
BRANCH = "main"  # or your feature branch

# Training: "fast" (Colab-friendly) or "full" (RF+GBM+LSTM, slower)
TRAIN_MODE = "fast"
LSTM_EPOCHS = 10  # only used when TRAIN_MODE == "full"

# Where YOU put the 12 CMAPSS .txt files on Colab SSD (session disk)
CMAPSS_UPLOAD_DIR = "/content/cmapss_upload"

# Databricks MLflow (PAT with scope: mlflow)
DATABRICKS_HOST = "https://dbc-xxxxxxxx.cloud.databricks.com"  # your workspace base URL only
DATABRICKS_TOKEN = ""  # paste PAT here, OR use Colab Secrets (next cell)
MLFLOW_EXPERIMENT = "/Shared/predictive_maintenance"
# ─────────────────────────────────────────────────────────────────────

## 1. Clone repository

In [ ]:
import os
import shutil
from pathlib import Path

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Set REPO_URL in the config cell to your real GitHub HTTPS URL.")

if Path(REPO_DIR).exists():
    print("Removing existing clone:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

!git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

print("Repo root:", Path.cwd())
print("Sample files:", list(Path("scripts").glob("*.py"))[:5])

## 2. Install dependencies

In [ ]:
# CUDA PyTorch for Colab GPU runtimes (safe on CPU too)
%pip install -q torch --index-url https://download.pytorch.org/whl/cu124
%pip install -q -r requirements.txt

In [ ]:
import torch
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "scripts" / "cmapss_colab_train.py").exists(), "Wrong directory — re-run clone cell"

print("Project:", ROOT)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: none (LSTM uses CPU; use TRAIN_MODE='fast' on free tier)")

## 3. Upload CMAPSS raw data (12 files)

Required files (from NASA zip `CMAPSSData/` or your PC `data/raw/cmapss/`):

`train_FD001.txt` `test_FD001.txt` `RUL_FD001.txt` … through **FD004** (12 files total).

**Option A — Colab file picker** (next cell): upload all `.txt` files.  
**Option B — Copy to SSD first:** put files in `/content/cmapss_upload/` via Colab left panel **Files → Upload**, then run **import** cell only.

In [ ]:
import shutil
from pathlib import Path

UPLOAD_DIR = Path(CMAPSS_UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
print("Upload folder:", UPLOAD_DIR)
print("Existing:", sorted(p.name for p in UPLOAD_DIR.glob("*.txt")))

In [ ]:
# Option A: pick files from your computer (run once; can select multiple .txt)
from google.colab import files

uploaded = files.upload()  # select all 12 train_/test_/RUL_ FD00X files
for name, data in uploaded.items():
    dest = UPLOAD_DIR / name
    dest.write_bytes(data)
    print(f"saved {name} ({len(data):,} bytes)")
print("In upload dir:", len(list(UPLOAD_DIR.glob("*.txt"))), "txt files")

In [ ]:
# Option B: unzip a NASA zip into the upload folder (skip if you uploaded .txt directly)
import zipfile

zips = list(UPLOAD_DIR.glob("*.zip")) + list(Path("/content").glob("*.zip"))
for zpath in zips:
    print("extract", zpath)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(UPLOAD_DIR)
print("txt in upload dir:", sorted(p.name for p in UPLOAD_DIR.rglob("*.txt"))[:15], "...")

In [ ]:
# Copy into project data/raw/cmapss/ (required for training)
!python scripts/import_cmapss_upload.py --source {CMAPSS_UPLOAD_DIR}

## 4. Connect MLflow → Databricks

Runs will appear in your workspace under **Machine Learning → Experiments**.

Optional: store PAT in Colab **Secrets** (key `DATABRICKS_TOKEN`) instead of pasting in config.

In [ ]:
import os

token = DATABRICKS_TOKEN
try:
    from google.colab import userdata
    if not token:
        token = userdata.get("DATABRICKS_TOKEN")
except Exception:
    pass

if not token:
    raise ValueError("Set DATABRICKS_TOKEN in config cell or Colab secret DATABRICKS_TOKEN")

import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "scripts/configure_mlflow_databricks.py",
        "--host",
        DATABRICKS_HOST,
        "--token",
        token,
        "--experiment",
        MLFLOW_EXPERIMENT,
        "--smoke-test",
    ],
    cwd=ROOT,
    check=True,
)

## 5. Build features (Phase 2) + train all FD subsets (Phase 3 → Databricks MLflow)

Writes:
- `data/processed/cmapss_FD00X_{train,test,predictions}.parquet`
- `artifacts/cmapss_*` + `artifacts/cmapss_training_registry.json`
- `models/*_FD00X.*`
- Databricks experiment `MLFLOW_EXPERIMENT` (runs: `FD00X_phase3_summary`)

In [ ]:
import os

# Propagate Databricks creds to subprocess (configure cell above)
os.environ["DATABRICKS_HOST"] = DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = token
os.environ["MLFLOW_TRACKING_URI"] = "databricks"
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT
os.environ["CMAPSS_UPLOAD_DIR"] = CMAPSS_UPLOAD_DIR

if TRAIN_MODE == "fast":
    !python scripts/cmapss_colab_train.py --fast --upload-dir {CMAPSS_UPLOAD_DIR} --mlflow-databricks
elif TRAIN_MODE == "full":
    !python scripts/cmapss_colab_train.py --lstm-epochs {LSTM_EPOCHS} --upload-dir {CMAPSS_UPLOAD_DIR} --mlflow-databricks
else:
    raise ValueError("TRAIN_MODE must be 'fast' or 'full'")

## 6. Supervisor verification (terminal + table)

In [ ]:
!python scripts/report_cmapss_mlflow.py

In [ ]:
import json
import os
from pathlib import Path

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

registry_path = Path("artifacts/cmapss_training_registry.json")
if registry_path.exists():
    print("Registry:")
    display(pd.json_normalize(json.loads(registry_path.read_text())["datasets"]).T)

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "databricks"))
client = MlflowClient()
exp_name = os.environ.get("MLFLOW_EXPERIMENT_NAME", MLFLOW_EXPERIMENT)
exp = client.get_experiment_by_name(exp_name)
if not exp:
    print("No MLflow experiment yet.")
else:
    runs = client.search_runs([exp.experiment_id], max_results=50)
    rows = []
    for r in runs:
        name = r.info.run_name or ""
        if not name.endswith("_phase3_summary"):
            continue
        rows.append({
            "dataset": r.data.params.get("dataset_id"),
            "winner": r.data.params.get("winner"),
            "skip_lstm": r.data.params.get("skip_lstm"),
            "gbm_max_rows": r.data.params.get("gbm_max_train_rows"),
            "test_rmse": r.data.metrics.get("test_rmse"),
            "test_nasa": r.data.metrics.get("test_rul_score"),
            "run_id": r.info.run_id,
        })
    display(pd.DataFrame(rows).sort_values("dataset"))

## 7. View runs in Databricks

Open your workspace → **Machine Learning → Experiments** → `predictive_maintenance` (or your `MLFLOW_EXPERIMENT` path).  
Local `mlflow ui` is only needed when `MLFLOW_TRACKING_URI=./mlruns`.

In [ ]:
print("Experiment:", os.environ.get("MLFLOW_EXPERIMENT_NAME"))
print("Workspace:", os.environ.get("DATABRICKS_HOST"))

## 8. Download artifacts to your laptop

In [ ]:
from pathlib import Path
from google.colab import files

required = ("models", "artifacts", "data/processed")
for p in required:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing {p}/ — run training cell first.")

zip_parts = "models artifacts data/processed"
if Path("mlruns").exists():
    zip_parts = "mlruns " + zip_parts
!zip -qr cmapss_colab_outputs.zip {zip_parts}
print("Zip size (MB):", round(Path("cmapss_colab_outputs.zip").stat().st_size / 1e6, 1))
files.download("cmapss_colab_outputs.zip")

## 9. On your PC (after unzip into project folder)

```bash
python scripts/report_cmapss_mlflow.py
mlflow ui --backend-store-uri ./mlruns --host 127.0.0.1 --port 5000
streamlit run dashboard/app.py
```

Dashboard sidebar lists every FD subset that has `cmapss_FD00X_predictions.parquet`.